In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Pre-processing

In [ ]:
news_dataset = pd.read_csv('fake_news_dataset_(3).csv')

In [ ]:
news_dataset.shape

(4499, 5)

In [ ]:
news_dataset.columns = news_dataset.columns.str.strip()

In [ ]:
print(news_dataset.columns)

Index(['id', 'title', 'author', 'text', 'label'], dtype='object')


In [ ]:
news_dataset.head()

,id,title,author,text,label
0,1,Politician Denies Corruption Allegations,David Green,Morning workouts are linked to improved mental...,1
1,2,NASA Confirms Discovery of New Planet,Isla Brown,The politician stated that the accusations wer...,1
2,3,Politician Denies Corruption Allegations,Bob Smith,An online post falsely claimed a common fruit ...,1
3,4,Fake News Article Sparks Public Outrage,Bob Smith,Astronomers have found a new celestial body or...,1
4,5,Weather Forecast: Heavy Rain Expected,Eva Adams,The politician stated that the accusations wer...,1


In [ ]:
news_dataset.isnull().sum()

,0
id,0
title,0
author,0
text,0
label,0


In [ ]:
news_dataset = news_dataset.fillna('')

In [ ]:
news_dataset['author'] = news_dataset['author'].fillna("")
news_dataset['title'] = news_dataset['title'].fillna("")
news_dataset['content'] = news_dataset['author'] + " " + news_dataset['title']

In [ ]:
print(news_dataset['content'])

0       David Green Politician Denies Corruption Alleg...
1        Isla Brown NASA Confirms Discovery of New Planet
2       Bob Smith Politician Denies Corruption Allegat...
3       Bob Smith Fake News Article Sparks Public Outrage
4         Eva Adams Weather Forecast: Heavy Rain Expected
                              ...                        
4494    Clara Lee Man Claims to Have Invented Time Mac...
4495    Frank Moore Politician Denies Corruption Alleg...
4496            Bob Smith Local Bakery Donates to Charity
4497    Henry Clark Study Shows Benefits of Morning Ex...
4498    Jack Turner Man Claims to Have Invented Time M...
Name: content, Length: 4499, dtype: object


In [ ]:
X = news_dataset.drop(columns='label', axis=1)
Y = news_dataset['label']

In [ ]:
print(X)
print(Y)

        id                                     title       author  \
0        1  Politician Denies Corruption Allegations  David Green   
1        2     NASA Confirms Discovery of New Planet   Isla Brown   
2        3  Politician Denies Corruption Allegations    Bob Smith   
3        4   Fake News Article Sparks Public Outrage    Bob Smith   
4        5     Weather Forecast: Heavy Rain Expected    Eva Adams   
...    ...                                       ...          ...   
4494  4495  Man Claims to Have Invented Time Machine    Clara Lee   
4495  4496  Politician Denies Corruption Allegations  Frank Moore   
4496  4497           Local Bakery Donates to Charity    Bob Smith   
4497  4498  Study Shows Benefits of Morning Exercise  Henry Clark   
4498  4499  Man Claims to Have Invented Time Machine  Jack Turner   

                                                   text  \
0     Morning workouts are linked to improved mental...   
1     The politician stated that the accusations wer.

Stemming:

In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [ ]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

In [ ]:
print(news_dataset['content'])

0            david green politician deni corrupt alleg
1         isla brown nasa confirm discoveri new planet
2              bob smith politician deni corrupt alleg
3       bob smith fake news articl spark public outrag
4          eva adam weather forecast heavi rain expect
                             ...                      
4494            clara lee man claim invent time machin
4495          frank moor politician deni corrupt alleg
4496              bob smith local bakeri donat chariti
4497       henri clark studi show benefit morn exercis
4498          jack turner man claim invent time machin
Name: content, Length: 4499, dtype: object


In [ ]:
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [ ]:
print(X)

['david green politician deni corrupt alleg'
 'isla brown nasa confirm discoveri new planet'
 'bob smith politician deni corrupt alleg' ...
 'bob smith local bakeri donat chariti'
 'henri clark studi show benefit morn exercis'
 'jack turner man claim invent time machin']


In [ ]:
print(Y)

[1 1 1 ... 0 1 0]


In [ ]:
Y.shape

(4499,)

In [ ]:
print(news_dataset.duplicated().sum())

0


In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [ ]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 31009 stored elements and shape (4499, 66)>
  Coords	Values
  (0, 2)	0.40852165676412067
  (0, 13)	0.40852165676412067
  (0, 16)	0.40770100798182496
  (0, 17)	0.40852165676412067
  (0, 31)	0.40770100798182496
  (0, 53)	0.40852165676412067
  (1, 7)	0.39484550859438106
  (1, 12)	0.3930622219390505
  (1, 19)	0.3930622219390505
  (1, 36)	0.39484550859438106
  (1, 47)	0.3930622219390505
  (1, 48)	0.26495736906724376
  (1, 51)	0.3930622219390505
  (2, 2)	0.40968570339385507
  (2, 6)	0.4053581735568735
  (2, 13)	0.40968570339385507
  (2, 17)	0.40968570339385507
  (2, 53)	0.40968570339385507
  (2, 57)	0.4053581735568735
  (3, 3)	0.2893262135304172
  (3, 6)	0.35695858713557177
  (3, 26)	0.36371732878830626
  (3, 49)	0.36371732878830626
  (3, 50)	0.36371732878830626
  (3, 54)	0.36371732878830626
  :	:
  (4495, 13)	0.4054103333558559
  (4495, 17)	0.4054103333558559
  (4495, 28)	0.4138658275535534
  (4495, 45)	0.4138658275535534
  (4495

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify=Y, random_state=2)

Model Training

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(X_train, Y_train)

LogisticRegression()

Evaluation

In [ ]:
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [ ]:
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.5398721867185329


In [ ]:
# accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [ ]:
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.4866666666666667


Predicting value


In [ ]:
X_new = X_test[3]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
  print('The news is Real')
else:
  print('The news is Fake')

[0]
The news is Real


In [ ]:
print(Y_test[3])

0


In [ ]:
rf_model = RandomForestClassifier()

In [ ]:
rf_model.fit(X_train, Y_train)

RandomForestClassifier()

In [ ]:
rf_test_pred = rf_model.predict(X_test)
rf_test_accuracy = accuracy_score(rf_test_pred, Y_test)

In [ ]:
print("Random Forest Test Accuracy:", rf_test_accuracy)

Random Forest Test Accuracy: 0.5088888888888888
